In [1]:
import nltk
import pandas as pd
import re

nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

df = pd.read_csv('spam_or_ham.csv')
df.head()

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


,v1,v2
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [2]:
df["lower"] = df["v2"].str.lower()
df[["v2", "lower"]].head()

,v2,lower
0,"Go until jurong point, crazy.. Available only ...","go until jurong point, crazy.. available only ..."
1,Ok lar... Joking wif u oni...,ok lar... joking wif u oni...
2,Free entry in 2 a wkly comp to win FA Cup fina...,free entry in 2 a wkly comp to win fa cup fina...
3,U dun say so early hor... U c already then say...,u dun say so early hor... u c already then say...
4,"Nah I don't think he goes to usf, he lives aro...","nah i don't think he goes to usf, he lives aro..."


In [3]:
df["no_punct"] = df["lower"].apply(lambda x: re.sub(r"[^\w\s]", " ", x))
df[["lower", "no_punct"]].head()

,lower,no_punct
0,"go until jurong point, crazy.. available only ...",go until jurong point crazy available only ...
1,ok lar... joking wif u oni...,ok lar joking wif u oni
2,free entry in 2 a wkly comp to win fa cup fina...,free entry in 2 a wkly comp to win fa cup fina...
3,u dun say so early hor... u c already then say...,u dun say so early hor u c already then say
4,"nah i don't think he goes to usf, he lives aro...",nah i don t think he goes to usf he lives aro...


In [4]:
df["lower"] = df["v2"].str.lower()
df["tokens"] = df["lower"].apply(word_tokenize)
df[["lower", "tokens"]].head()

,lower,tokens
0,"go until jurong point, crazy.. available only ...","[go, until, jurong, point, ,, crazy, .., avail..."
1,ok lar... joking wif u oni...,"[ok, lar, ..., joking, wif, u, oni, ...]"
2,free entry in 2 a wkly comp to win fa cup fina...,"[free, entry, in, 2, a, wkly, comp, to, win, f..."
3,u dun say so early hor... u c already then say...,"[u, dun, say, so, early, hor, ..., u, c, alrea..."
4,"nah i don't think he goes to usf, he lives aro...","[nah, i, do, n't, think, he, goes, to, usf, ,,..."


In [5]:
stop_words = set(stopwords.words("english"))

df["no_stop"] = df["tokens"].apply(lambda words: [w for w in words if w not in stop_words])
df[["tokens", "no_stop"]].head()

,tokens,no_stop
0,"[go, until, jurong, point, ,, crazy, .., avail...","[go, jurong, point, ,, crazy, .., available, b..."
1,"[ok, lar, ..., joking, wif, u, oni, ...]","[ok, lar, ..., joking, wif, u, oni, ...]"
2,"[free, entry, in, 2, a, wkly, comp, to, win, f...","[free, entry, 2, wkly, comp, win, fa, cup, fin..."
3,"[u, dun, say, so, early, hor, ..., u, c, alrea...","[u, dun, say, early, hor, ..., u, c, already, ..."
4,"[nah, i, do, n't, think, he, goes, to, usf, ,,...","[nah, n't, think, goes, usf, ,, lives, around,..."


In [6]:
lemmatizer = WordNetLemmatizer()

df["lemma"] = df["no_stop"].apply(lambda words: [lemmatizer.lemmatize(w) for w in words])
df[["no_stop", "lemma"]].head()

,no_stop,lemma
0,"[go, jurong, point, ,, crazy, .., available, b...","[go, jurong, point, ,, crazy, .., available, b..."
1,"[ok, lar, ..., joking, wif, u, oni, ...]","[ok, lar, ..., joking, wif, u, oni, ...]"
2,"[free, entry, 2, wkly, comp, win, fa, cup, fin...","[free, entry, 2, wkly, comp, win, fa, cup, fin..."
3,"[u, dun, say, early, hor, ..., u, c, already, ...","[u, dun, say, early, hor, ..., u, c, already, ..."
4,"[nah, n't, think, goes, usf, ,, lives, around,...","[nah, n't, think, go, usf, ,, life, around, th..."


In [7]:
df["clean_text"] = df["lemma"].apply(lambda x: " ".join(x))
df[["clean_text"]].head()

,clean_text
0,"go jurong point , crazy .. available bugis n g..."
1,ok lar ... joking wif u oni ...
2,free entry 2 wkly comp win fa cup final tkts 2...
3,u dun say early hor ... u c already say ...
4,"nah n't think go usf , life around though"


In [10]:
df=df[['v1','clean_text']]

In [12]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

In [13]:



df = df.rename(columns={"v1": "label", "clean_text": "content"})

vectorizer = CountVectorizer(stop_words='english', max_features=1500)
X = vectorizer.fit_transform(df['content'])


y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [14]:


model = SVC(kernel='linear')
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))


Accuracy: 0.979372197309417


In [16]:
from sklearn.metrics import classification_report, confusion_matrix

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

[[958   7]
 [ 16 134]]
              precision    recall  f1-score   support

         ham       0.98      0.99      0.99       965
        spam       0.95      0.89      0.92       150

    accuracy                           0.98      1115
   macro avg       0.97      0.94      0.95      1115
weighted avg       0.98      0.98      0.98      1115



In [24]:
new_message = ["hi"]

new_vec = vectorizer.transform(new_message)

prediction = model.predict(new_vec)
print("Prediction:", prediction[0])


Prediction: ham


In [19]:
import joblib

joblib.dump(model, 'spam_model.pkl')

joblib.dump(vectorizer, 'vectorizer.pkl')

['vectorizer.pkl']